# Regression Tree Impurity: MSE, Variance, and MAE

---

## Why Gini and Entropy Do Not Work for Regression

Gini impurity and entropy are designed for **classification**, where the target $y$ takes one of a finite set of class labels (for example, $\{0,1,2\}$). They are defined in terms of **class probabilities** $p_i$ and measure how mixed the classes are.

In **regression**, the target $y$ is **continuous**, so:
- There is no finite set of discrete classes.
- There are no well-defined class probabilities $p_i$ to plug into Gini or entropy.

Instead, in regression trees we care about how **spread out** the continuous target values are within a node. A natural way to measure this spread is by using the **variance** of the target values in the node.

---

## Mean Squared Error (MSE) as Node Impurity

Consider a node that contains $n$ samples with continuous target values
$$
y_1, y_2, \dots, y_n.
$$

Suppose the leaf prediction for this node is a **constant** value $c$ (the same prediction for every sample in the node). The **Mean Squared Error (MSE)** of this node with prediction $c$ is
$$
\text{MSE}(c) = \frac{1}{n} \sum_{i=1}^{n} (y_i - c)^2.
$$

This is the impurity measure used in regression trees when we use squared error.

### Optimal Node Prediction Under MSE Is the Mean

We want to find the value $c$ that **minimizes** $\text{MSE}(c)$. Define the unscaled sum of squared errors
$$
S(c) = \sum_{i=1}^{n} (y_i - c)^2.
$$

Minimizing $S(c)$ or $\text{MSE}(c)$ is equivalent (they differ only by the factor $\frac{1}{n}$).

Take the derivative of $S(c)$ with respect to $c$:
$$
S(c) = \sum_{i=1}^{n} (y_i - c)^2
$$
$$
\frac{dS}{dc} = \sum_{i=1}^{n} 2 (y_i - c)(-1)
$$
$$
\frac{dS}{dc} = -2 \sum_{i=1}^{n} (y_i - c).
$$

Set the derivative equal to zero:
$$
-2 \sum_{i=1}^{n} (y_i - c) = 0
$$
$$
\sum_{i=1}^{n} (y_i - c) = 0
$$
$$
\sum_{i=1}^{n} y_i - n c = 0
$$
$$
n c = \sum_{i=1}^{n} y_i
$$
$$
c^\star = \frac{1}{n} \sum_{i=1}^{n} y_i.
$$

So the optimal constant prediction $c^\star$ that minimizes MSE is the **sample mean**
$$
\boxed{c^\star = \bar{y} = \frac{1}{n} \sum_{i=1}^{n} y_i.}
$$

This is why regression trees predict the **average** target value in each leaf when using MSE.

---

## Variance Reduction as the Split Criterion

For a node with $n$ samples and mean $\bar{y}$, if we choose $c = \bar{y}$, then
$$
\text{MSE}(\bar{y}) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \bar{y})^2,
$$
which is exactly the **sample variance** up to a constant factor. So the node impurity under MSE is proportional to the variance of the targets in the node.

Now consider a split of a parent node $P$ into two child nodes $L$ (left) and $R$ (right).

- Parent node:
  - $n_P$ samples
  - targets $y_i$ for $i \in P$
  - mean $\bar{y}_P$
  - impurity:
    $$
    \text{Imp}(P) = \frac{1}{n_P} \sum_{i \in P} (y_i - \bar{y}_P)^2.
    $$

- Left child:
  - $n_L$ samples
  - mean $\bar{y}_L$
  - impurity:
    $$
    \text{Imp}(L) = \frac{1}{n_L} \sum_{i \in L} (y_i - \bar{y}_L)^2.
    $$

- Right child:
  - $n_R$ samples
  - mean $\bar{y}_R$
  - impurity:
    $$
    \text{Imp}(R) = \frac{1}{n_R} \sum_{i \in R} (y_i - \bar{y}_R)^2.
    $$

with
$$
n_P = n_L + n_R.
$$

### Variance Reduction Formula

The **split criterion** is to maximize the **reduction in impurity**:
$$
\Delta = \text{Imp}(P) - \left( \frac{n_L}{n_P} \,\text{Imp}(L) + \frac{n_R}{n_P} \,\text{Imp}(R) \right).
$$

Here:
- $\text{Imp}(P)$ is the parent impurity (variance-like MSE).
- $\text{Imp}(L)$ and $\text{Imp}(R)$ are the child impurities.
- $\frac{n_L}{n_P}$ and $\frac{n_R}{n_P}$ are weights (proportion of samples in each child).

Because each node’s impurity under MSE equals its **optimal** MSE (with prediction equal to the node mean), this $\Delta$ is exactly the **reduction in MSE** produced by the split.

So:
$$
\boxed{
\Delta
= \text{Var}(P)
- \left(
\frac{n_L}{n_P} \,\text{Var}(L)
+ \frac{n_R}{n_P} \,\text{Var}(R)
\right)
}
$$
is the same as maximizing the **MSE reduction** from parent to children.

---

## Why the Mean Is the Leaf Prediction Under MSE (Alternate Proof)

Consider again
$$
S(c) = \sum_{i=1}^{n} (y_i - c)^2,
$$
and let $\bar{y}$ be the sample mean:
$$
\bar{y} = \frac{1}{n} \sum_{i=1}^{n} y_i.
$$

Rewrite $y_i - c$ as $(y_i - \bar{y}) + (\bar{y} - c)$:
$$
\begin{aligned}
S(c)
&= \sum_{i=1}^{n} \big[(y_i - \bar{y}) + (\bar{y} - c)\big]^2 \\
&= \sum_{i=1}^{n} (y_i - \bar{y})^2
   + 2(\bar{y} - c)\sum_{i=1}^{n} (y_i - \bar{y})
   + \sum_{i=1}^{n} (\bar{y} - c)^2.
\end{aligned}
$$

But
$$
\sum_{i=1}^{n} (y_i - \bar{y}) = 0,
$$
so the middle term is zero. We get
$$
S(c) = \sum_{i=1}^{n} (y_i - \bar{y})^2 + n(\bar{y} - c)^2.
$$

The first term does not depend on $c$. The second term is minimized when $(\bar{y} - c)^2$ is minimized, which occurs at
$$
\boxed{c = \bar{y}.}
$$

This shows again that the mean is the unique minimizer of the sum of squared errors over all constants.

---

## Mean Absolute Error (MAE) as an Alternative Impurity

Instead of squared error, we can use **absolute error** as the impurity measure.

For a node with $n$ samples and constant prediction $c$, the **Mean Absolute Error (MAE)** is
$$
\text{MAE}(c) = \frac{1}{n} \sum_{i=1}^{n} |y_i - c|.
$$

This is an $L_1$-type impurity (absolute deviation).

### Optimal Leaf Prediction Under MAE Is the Median

The key fact is:

> Among all constants $c$, the value that minimizes $\text{MAE}(c)$ is a **median** of the $y_i$ values.

To see why, sort the target values:
$$
y_{(1)} \le y_{(2)} \le \dots \le y_{(n)}.
$$

Consider the function
$$
f(c) = \sum_{i=1}^{n} |y_i - c|.
$$

As $c$ moves from $-\infty$ to $+\infty$, each time $c$ passes a data point $y_{(k)}$, the slope of $f(c)$ changes by $+2$ or $-2$ because one term changes from $(y_{(k)} - c)$ to $(c - y_{(k)})$ or vice versa.

- When more than half the data is to the **right** of $c$, the slope is negative, so $f(c)$ decreases as $c$ moves right.
- When more than half the data is to the **left** of $c$, the slope is positive, so $f(c)$ increases as $c moves right.
- At a **median** (a value where at least half the data is on each side), the slope crosses zero or is flat.

Therefore, $f(c)$ is minimized at any median of the data. So for MAE impurity, the optimal leaf prediction is the **median** of the target values:
$$
\boxed{c^\star = \text{median}(y_1, \dots, y_n).}
$$

---

## MSE vs MAE: Comparison

### Impurity Formulas

- **MSE impurity**:
  $$
  \text{MSE}(c) = \frac{1}{n} \sum_{i=1}^{n} (y_i - c)^2.
  $$

- **MAE impurity**:
  $$
  \text{MAE}(c) = \frac{1}{n} \sum_{i=1}^{n} |y_i - c|.
  $$

### Optimal Leaf Prediction

- Under MSE: optimal $c$ is the **mean** $\bar{y}$.
- Under MAE: optimal $c$ is a **median** of $\{y_i\}$.

### Sensitivity to Outliers

- MSE (squared error) **penalizes large errors strongly**, so it is very **sensitive to outliers**.
- MAE (absolute error) grows **linearly** with error size, so it is **more robust to outliers**.

### Computational Aspects

- MSE (with optimal prediction):
  - Compute the mean (one pass), then sum squared deviations (another pass).
- MAE (with optimal prediction):
  - Need a median (via selection or sorting), then sum absolute deviations.

### Summary Table

| Aspect                  | MSE                                   | MAE                                        |
|-------------------------|----------------------------------------|--------------------------------------------|
| Impurity formula        | $\frac{1}{n}\sum (y_i - c)^2$         | $\frac{1}{n}\sum |y_i - c|$                |
| Optimal leaf prediction | Mean of $y_i$                         | Median of $y_i$                            |
| Loss type               | Squared error ($L_2$)                 | Absolute error ($L_1$)                     |
| Outlier sensitivity     | High (heavily influenced by outliers) | Lower (more robust to outliers)            |
| When preferred          | When large errors must be penalized   | When robustness to outliers is important   |

---

## Numerical Example: Variance Reduction by Hand

Consider a parent node with the following 6 target values:
$$
\{2,\ 3,\ 4,\ 10,\ 11,\ 12\}.
$$

Suppose we split so that:

- Left child: $\{2, 3, 4\}$  
- Right child: $\{10, 11, 12\}$

### Step 1: Parent Mean and Impurity

Number of samples:
$$
n_P = 6.
$$

Parent mean:
$$
\bar{y}_P = \frac{2 + 3 + 4 + 10 + 11 + 12}{6} = \frac{42}{6} = 7.
$$

Parent impurity:
$$
\begin{aligned}
\text{Imp}(P)
  &= \frac{1}{6} \Big[(2-7)^2 + (3-7)^2 + (4-7)^2 + (10-7)^2 + (11-7)^2 + (12-7)^2\Big] \\
  &= \frac{1}{6} \Big[25 + 16 + 9 + 9 + 16 + 25\Big] \\
  &= \frac{1}{6} \cdot 100 \\
  &= \frac{100}{6} \approx 16.67.
\end{aligned}
$$

### Step 2: Left Child Mean and Impurity

Left child: $y_L = \{2, 3, 4\}$, $n_L = 3$.

Mean:
$$
\bar{y}_L = \frac{2 + 3 + 4}{3} = \frac{9}{3} = 3.
$$

Impurity:
$$
\begin{aligned}
\text{Imp}(L)
  &= \frac{1}{3} \Big[(2-3)^2 + (3-3)^2 + (4-3)^2\Big] \\
  &= \frac{1}{3} \Big[1 + 0 + 1\Big] \\
  &= \frac{2}{3} \approx 0.67.
\end{aligned}
$$

### Step 3: Right Child Mean and Impurity

Right child: $y_R = \{10, 11, 12\}$, $n_R = 3$.

Mean:
$$
\bar{y}_R = \frac{10 + 11 + 12}{3} = \frac{33}{3} = 11.
$$

Impurity:
$$
\begin{aligned}
\text{Imp}(R)
  &= \frac{1}{3} \Big[(10-11)^2 + (11-11)^2 + (12-11)^2\Big] \\
  &= \frac{1}{3} \Big[1 + 0 + 1\Big] \\
  &= \frac{2}{3} \approx 0.67.
\end{aligned}
$$

### Step 4: Weighted Child Impurity

Total samples:
$$
n_P = n_L + n_R = 6.
$$

Weighted impurity:
$$
\begin{aligned}
\text{Imp}_{\text{children}}
  &= \frac{n_L}{n_P} \,\text{Imp}(L) + \frac{n_R}{n_P} \,\text{Imp}(R) \\
  &= \frac{3}{6} \cdot \frac{2}{3} + \frac{3}{6} \cdot \frac{2}{3} \\
  &= \frac{1}{2} \cdot \frac{2}{3} + \frac{1}{2} \cdot \frac{2}{3} \\
  &= \frac{2}{3} \approx 0.67.
\end{aligned}
$$

### Step 5: Variance (MSE) Reduction

$$
\begin{aligned}
\Delta
  &= \text{Imp}(P) - \text{Imp}_{\text{children}} \\
  &= \frac{100}{6} - \frac{2}{3} \\
  &= \frac{100}{6} - \frac{4}{6} \\
  &= \frac{96}{6} \\
  &= 16.
\end{aligned}
$$

The split reduces the node impurity by $16$ (in MSE terms). A regression tree would compare this reduction with other possible splits and choose the split with the **largest** impurity reduction.